# Two-Stream ST-GCN — Depth Search on FX Subset

Mục tiêu: tìm độ sâu (số block) phù hợp nhất cho Two-Stream ST-GCN trên tập FX (35 classes), nơi mô hình depth=10 bị overfit.

| Depth | Blocks per stream | ~Params (2 streams) | Ghi chú |
|-------|------------------|---------------------|---------|
| 10    | 64×4 → 128×3 → 256×3 | ~6.2M | Baseline, đang overfit |
| 8     | 64×3 → 128×2 → 256×3 | ~5.5M | Giảm nhẹ |
| 6     | 64×2 → 128×2 → 256×2 | ~4.5M | Compact |
| 4     | 64×1 → 128×1 → 256×2 | ~3.2M | Lightweight |

**Pipeline:**
```
Gym288 → build Gym99 → lọc FX (35 cls) → train 4 variants → so sánh train/val curves
```

**Config chung cho tất cả depths:**
- Apparatus: FX, 35 classes (label 6–40 → local 0–34)
- Joint spec: COCO18
- Loss: Focal (sqrt_inverse alpha)
- Augmentation: SkeletonFeeder + WeightedRandomSampler
- Normalization: center_norm
- Model: `--use_two_stream` với từng `--model_depth`

In [ ]:
# ── Cell 1: Environment & paths ───────────────────────────────────────────────
import os
import subprocess
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'], check=True)
    from dotenv import load_dotenv

_nb_dir = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
for _candidate in [_nb_dir / '.env', _nb_dir.parent / '.env', Path('.env')]:
    if _candidate.exists():
        load_dotenv(_candidate)
        print(f'Loaded .env from {_candidate}')
        break
else:
    print('No .env found — using Kaggle defaults')

REPO_DIR   = os.getenv('REPO_DIR',   '/kaggle/working/Yolo-ST-GCN')
BRANCH     = os.getenv('BRANCH',     'experiment-bonestream')
GYM288_PKL = os.getenv('GYM288_PKL', '/kaggle/working/Gym288-skeleton/gym_288_skeleton.pkl')
GYM99_PKL  = os.getenv('GYM99_PKL',  '/kaggle/working/Gym99-from-Gym288/gym99_from_gym288.pkl')
OUT_BASE   = os.getenv('OUT_DIR',    'outputs/depth_search_fx')

# ── Experiment config ─────────────────────────────────────────────────────────
DEPTHS                  = [8, 6, 4]   # depth=10 already trained separately
EPOCHS                  = 50
BATCH_SIZE              = 128
LR                      = '0.001'
WARMUP_EPOCHS           = '5'
EARLY_STOPPING_PATIENCE = '10'        # stop if val_loss stagnates for 10 epochs (0 = off)
POLICY_PATH             = '/kaggle/working/fx_aug_policy.json'

DEPTH_OUT_DIRS = {d: f'{OUT_BASE}/depth_{d}_2s' for d in DEPTHS}

print(f'REPO_DIR                  = {REPO_DIR}')
print(f'GYM288_PKL                = {GYM288_PKL}')
print(f'GYM99_PKL                 = {GYM99_PKL}')
print(f'OUT_BASE                  = {OUT_BASE}')
print(f'Depths to run             = {DEPTHS}')
print(f'Early stopping patience   = {EARLY_STOPPING_PATIENCE}')

In [ ]:
# ── Cell 2: Repo setup ────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'

if not Path(REPO_DIR).exists():
    print('Cloning repo...')
    subprocess.run(
        ['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, REPO_DIR],
        check=True,
    )
else:
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Working dir:', os.getcwd())

In [ ]:
# ── Cell 3: Download Gym288 dataset ───────────────────────────────────────────
if Path(GYM288_PKL).exists():
    print(f'Gym288 pickle found: {GYM288_PKL}')
else:
    print('Downloading from HuggingFace...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'], check=True)
    from huggingface_hub import snapshot_download
    download_dir = Path(GYM288_PKL).parent
    download_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='Lozumi/Gym288-skeleton',
        repo_type='dataset',
        local_dir=str(download_dir),
        local_dir_use_symlinks=False,
    )
    pkl_candidates = sorted(download_dir.rglob('*.pkl'))
    if not pkl_candidates:
        raise FileNotFoundError('No .pkl found after Gym288 download.')
    GYM288_PKL = str(pkl_candidates[0])
    print(f'Downloaded: {GYM288_PKL}')

In [ ]:
# ── Cell 4: Custom augmentation policy for FX ─────────────────────────────────
import json

fx_policy = {
    "0": {
        "horizontal_flip_prob": 0.5,
        "scale_prob": 0.2,
        "scale_range": [0.95, 1.05],
        "random_shift": False,
        "random_move": False,
        "noise_std": 0.0,
        "joint_drop_prob": 0.0,
        "subsample_prob": 0.0
    },
    "1": {
        "horizontal_flip_prob": 0.5,
        "scale_prob": 0.5,
        "scale_range": [0.9, 1.1],
        "random_shift": True,
        "random_move": True,
        "move_angle": 5.0,
        "move_scale": 0.05,
        "move_trans": 0.0,
        "noise_std": 0.005,
        "joint_drop_prob": 0.05,
        "subsample_prob": 0.3,
        "subsample_factor_range": [0.9, 1.1]
    },
    "2": {
        "horizontal_flip_prob": 0.5,
        "scale_prob": 0.8,
        "scale_range": [0.85, 1.15],
        "random_shift": True,
        "random_move": True,
        "move_angle": 10.0,
        "move_scale": 0.1,
        "move_trans": 0.0,
        "noise_std": 0.01,
        "joint_drop_prob": 0.1,
        "subsample_prob": 0.5,
        "subsample_factor_range": [0.8, 1.2],
        "temporal_reverse_prob": 0.0
    }
}

Path(POLICY_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(POLICY_PATH, 'w') as f:
    json.dump(fx_policy, f, indent=4)

print(f'Augmentation policy saved: {POLICY_PATH}')

In [ ]:
# ── Cell 5: Train Two-Stream depth=8 ─────────────────────────────────────────
import importlib

DEPTH = 8
OUT   = DEPTH_OUT_DIRS[DEPTH]

_dsuffix   = f'_d{DEPTH}' if DEPTH != 10 else ''
checkpoint = Path(OUT) / f'stgcn_gym99_coco18_2s{_dsuffix}_expert_FX.pth'
if checkpoint.exists():
    print(f'[depth={DEPTH}] Checkpoint found, skipping training: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--warmup_epochs',           WARMUP_EPOCHS,
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--use_two_stream',
        '--model_depth',             str(DEPTH),
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--center_norm',
        '--use_augment_feeder',
        '--aug_config_path',         POLICY_PATH,
        '--use_weighted_sampler',
        '--grad_clip_norm',          '1.0',
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print(f'>>> Training Two-Stream depth={DEPTH} on FX (35 classes)...')
    _train.main()
    print(f'\n✅ depth={DEPTH} done.')

In [ ]:
# ── Cell 6: Train Two-Stream depth=6 ─────────────────────────────────────────
import importlib

DEPTH = 6
OUT   = DEPTH_OUT_DIRS[DEPTH]

_dsuffix   = f'_d{DEPTH}' if DEPTH != 10 else ''
checkpoint = Path(OUT) / f'stgcn_gym99_coco18_2s{_dsuffix}_expert_FX.pth'
if checkpoint.exists():
    print(f'[depth={DEPTH}] Checkpoint found, skipping training: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--warmup_epochs',           WARMUP_EPOCHS,
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--use_two_stream',
        '--model_depth',             str(DEPTH),
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--center_norm',
        '--use_augment_feeder',
        '--aug_config_path',         POLICY_PATH,
        '--use_weighted_sampler',
        '--grad_clip_norm',          '1.0',
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print(f'>>> Training Two-Stream depth={DEPTH} on FX (35 classes)...')
    _train.main()
    print(f'\n✅ depth={DEPTH} done.')

In [ ]:
# ── Cell 7: Train Two-Stream depth=4 ─────────────────────────────────────────
import importlib

DEPTH = 4
OUT   = DEPTH_OUT_DIRS[DEPTH]

_dsuffix   = f'_d{DEPTH}' if DEPTH != 10 else ''
checkpoint = Path(OUT) / f'stgcn_gym99_coco18_2s{_dsuffix}_expert_FX.pth'
if checkpoint.exists():
    print(f'[depth={DEPTH}] Checkpoint found, skipping training: {checkpoint}')
else:
    sys.argv = [
        'train_gym99.py',
        '--auto_build_from_gym288',
        '--gym288_dataset_path',     GYM288_PKL,
        '--dataset_path',            GYM99_PKL,
        '--out_dir',                 OUT,
        '--apparatus',               'FX',
        '--epochs',                  str(EPOCHS),
        '--batch_size',              str(BATCH_SIZE),
        '--lr',                      LR,
        '--warmup_epochs',           WARMUP_EPOCHS,
        '--early_stopping_patience', EARLY_STOPPING_PATIENCE,
        '--num_workers',             '2',
        '--joint_spec_name',         'coco18',
        '--use_two_stream',
        '--model_depth',             str(DEPTH),
        '--loss_name',               'focal',
        '--focal_alpha_mode',        'sqrt_inverse',
        '--center_norm',
        '--use_augment_feeder',
        '--aug_config_path',         POLICY_PATH,
        '--use_weighted_sampler',
        '--grad_clip_norm',          '1.0',
        '--weight_decay',            '0.0005',
        '--save_every_epochs',       '10',
    ]
    import scripts.train_gym99 as _train
    importlib.reload(_train)
    print(f'>>> Training Two-Stream depth={DEPTH} on FX (35 classes)...')
    _train.main()
    print(f'\n✅ depth={DEPTH} done.')

In [ ]:
# ── Cell 8: So sánh training curves tất cả depths ────────────────────────────
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

COLORS = {8: '#e67e22', 6: '#27ae60', 4: '#2980b9'}
STYLES = {8: '--',      6: '-.',      4: ':'}

histories = {}
for d in DEPTHS:
    h_path = Path(DEPTH_OUT_DIRS[d]) / 'history.json'
    if h_path.exists():
        with open(h_path) as f:
            histories[d] = json.load(f)
        print(f'[depth={d}] Loaded history ({len(histories[d]["val_acc"])} epochs)')
    else:
        print(f'[depth={d}] history.json not found, skipping.')

if not histories:
    print('No histories loaded — run the training cells first.')
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Two-Stream ST-GCN — Depth Search on FX (35 classes)', fontsize=14, fontweight='bold')

    plot_cfg = [
        (axes[0, 0], 'train_loss', 'Train Loss'),
        (axes[0, 1], 'val_loss',   'Val Loss'),
        (axes[1, 0], 'train_acc',  'Train Accuracy'),
        (axes[1, 1], 'val_acc',    'Val Accuracy'),
    ]

    for ax, key, title in plot_cfg:
        for d, h in sorted(histories.items(), reverse=True):
            if key in h:
                epochs = range(1, len(h[key]) + 1)
                ax.plot(epochs, h[key],
                        color=COLORS[d], linestyle=STYLES[d],
                        linewidth=1.8, label=f'depth={d}')
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=10)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    plt.tight_layout()
    out_png = Path(OUT_BASE) / 'depth_search_curves.png'
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_png}')

In [ ]:
# ── Cell 10: Overfitting gap — train acc vs val acc ───────────────────────────
# Plot riêng để dễ thấy gap giữa train và val cho từng depth

if not histories:
    print('No histories loaded — run the training cells first.')
else:
    n_depths = len(histories)
    fig, axes = plt.subplots(1, n_depths, figsize=(5 * n_depths, 4), sharey=True)
    if n_depths == 1:
        axes = [axes]
    fig.suptitle('Train vs Val Accuracy per Depth (Overfitting Gap)', fontsize=13, fontweight='bold')

    for ax, (d, h) in zip(axes, sorted(histories.items())):
        epochs = range(1, len(h['train_acc']) + 1)
        ax.plot(epochs, h['train_acc'], color='steelblue', linewidth=1.8, label='Train')
        ax.plot(epochs, h['val_acc'],   color='tomato',    linewidth=1.8, label='Val')
        ax.fill_between(epochs, h['val_acc'], h['train_acc'],
                        alpha=0.15, color='orange', label='Gap')

        best_val   = max(h['val_acc'])
        best_epoch = h['val_acc'].index(best_val) + 1
        ax.axvline(best_epoch, color='gray', linestyle='--', linewidth=1, alpha=0.7)
        ax.set_title(f'depth={d}  (best val={best_val:.3f} @ ep{best_epoch})', fontsize=11)
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    axes[0].set_ylabel('Accuracy')
    plt.tight_layout()
    out_png = Path(OUT_BASE) / 'depth_search_gap.png'
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_png}')

In [ ]:
# ── Cell 11: Bảng tổng kết kết quả ──────────────────────────────────────────
import json

print(f'{'Depth':>6}  {'Best Val Acc':>12}  {'@Epoch':>7}  {'Final Val Acc':>13}  {'Final Train Acc':>15}  {'Gap (final)':>11}  {'Macro F1':>9}')
print('-' * 82)

for d in sorted(histories.keys()):
    h = histories[d]
    best_val   = max(h['val_acc'])
    best_epoch = h['val_acc'].index(best_val) + 1
    final_val  = h['val_acc'][-1]
    final_tr   = h['train_acc'][-1]
    gap        = final_tr - final_val

    # Load macro F1 from metrics file if available
    m_path = Path(DEPTH_OUT_DIRS[d]) / 'metrics_train_gym99.json'
    macro_f1 = '—'
    if m_path.exists():
        with open(m_path) as f:
            m = json.load(f)
        macro_f1 = f"{m.get('macro_f1', 0):.4f}"

    print(f'{d:>6}  {best_val:>12.4f}  {best_epoch:>7}  {final_val:>13.4f}  {final_tr:>15.4f}  {gap:>+11.4f}  {macro_f1:>9}')